In [1]:
import logging
import osmnx as ox
import geopandas as gpd
import networkx as nx
from shapely.geometry import Polygon
import numpy as np
import pandas as pd

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s"
)
logger = logging.getLogger("accessibility_pipeline")


In [ ]:
# This is where I will define cities and categories
cell_size = 75  # meters (edge-to-edge spacing concept)
cities = [
    "Canberra, Australian Capital Territory, Australia",
    "Darwin, Northern Territory, Australia",
    "Adelaide, South Australia, Australia",
    "Hobart, Tasmania, Australia",
]

categories = [
    {
        "id": "grocery",
        "usePolygons": False,
        "tags": {
            "shop": [
                "supermarket",
                "grocery",
                "greengrocer"
            ]
        }
    },

    {
        "id": "healthcare",
        "usePolygons": False,
        "tags": {
            "amenity": [
                "hospital",
                "clinic",
                "doctors",
                "pharmacy"
            ],
            "healthcare": [
                "clinic",
                "doctor",
                "hospital"
            ]
        }
    },

    {
        "id": "education",
        "usePolygons": True,
        "tags": {
            "amenity": [
                "school",
                "university",
                "college"
            ]
        }
    },
    {
        "id": "transit",
        "usePolygons": False,
        "tags": {
            "amenity": [
                "bus_station",
                "ferry_terminal"
            ],
            "public_transport": [
                "station",
                "stop_position"
            ],
            "railway": [
                "station",
                "tram_stop",
                "subway_entrance"
            ]
        }
    },
    {
        "id": "libraries",
        "usePolygons": False,
        "tags": {
            "amenity": [
                "library"
            ]
        }
    },

    {
        "id": "parks",
        "usePolygons": True,
        "tags": {
            "leisure": [
                "park",
                "garden",
                "nature_reserve",
                "playground"
            ],
            "landuse": [
                "recreation_ground"
            ]
        }
    }
]

In [3]:
def load_city(city_name):
    """
    Returns:
        boundary (GeoDataFrame)
        graph (networkx graph in projected CRS)
        crs (string)
    """

    logger.info(f"Loading city data for {city_name}")
    place_name = city_name
    
    boundary = ox.geocode_to_gdf(place_name)
    logger.debug(f"Boundary loaded: {boundary.shape[0]} feature(s)")

    G = ox.graph_from_place(place_name, network_type="walk")
    G = ox.project_graph(G)
    logger.debug(
        f"Graph loaded: {len(G.nodes())} nodes, {len(G.edges())} edges"
    )
    
    target_crs = G.graph["crs"]
    
    boundary = boundary.to_crs(target_crs)
    logger.debug(f"Boundary reprojected to {target_crs}")

    water_tags = {
        "natural": "water"
    }
    
    water = ox.features_from_place(
        city_name,
        water_tags
    )
    
    water = water.to_crs(target_crs)
    water_union = water.union_all()
    logger.debug("Water features loaded and unioned")

    boundary["geometry"] = boundary.geometry.difference(
        water_union
    )
    logger.info("City boundary cleaned by removing water areas")

    return boundary, G, target_crs


In [4]:
def create_hex_grid(boundary, cell_size, crs):
    """
    Returns hexagonal grid clipped to city boundary
    """

    logger.info(
        f"Creating hex grid for boundary with cell size {cell_size} meters"
    )

    def hexagon(center_x, center_y, r):
        # pointy-top orientation
        angles = np.arange(0, 360, 60) + 30  # critical: rotation = 30°
        
        return Polygon([
            (
                center_x + r * np.cos(np.radians(a)),
                center_y + r * np.sin(np.radians(a))
            )
            for a in angles
        ])
    
    
    def create_hex_grid_internal(boundary_gdf, cell_size):
        minx, miny, maxx, maxy = boundary_gdf.total_bounds
    
        r = cell_size
    
        dx = np.sqrt(3) * r
        dy = 1.5 * r
    
        polygons = []
    
        row = 0
        y = miny
    
        while y <= maxy + dy:
    
            # IMPORTANT: offset depends on row
            x_offset = (dx / 2) if row % 2 else 0
            x = minx + x_offset
    
            while x <= maxx + dx:
                polygons.append(hexagon(x, y, r))
                x += dx
    
            y += dy
            row += 1
    
        return gpd.GeoDataFrame(geometry=polygons, crs=boundary_gdf.crs)

    hex_grid = create_hex_grid_internal(boundary, cell_size)
    logger.debug(
        f"Generated {len(hex_grid)} hexagons before clipping"
    )

    hex_grid = hex_grid.set_crs(crs)

    hex_grid = gpd.overlay(hex_grid, boundary, how="intersection")
    logger.debug(
        f"Hex grid clipped to boundary: {len(hex_grid)} cells remain"
    )

    hex_grid["centroid"] = hex_grid.geometry.centroid
    logger.info("Hex grid created and centroids computed")

    return hex_grid


In [5]:
def create_points_along_line(line, spacing=50):
    """
    Create points every <spacing> metres along a LineString.
    """
    distances = np.arange(0, line.length, spacing)

    return [
        line.interpolate(distance)
        for distance in distances
    ]

In [6]:
def load_pois(city_name, category, target_crs):
    """
    Returns POIs for a category based on OSM tags
    """

    category_id = category["id"]
    logger.info(f"Loading POIs for category '{category_id}'")

    tags = category["tags"]
    use_polygons = category["usePolygons"]

    pois = ox.features_from_place(city_name, tags)
    pois = pois.to_crs(target_crs)
    logger.debug(f"Raw POI count for {category_id}: {len(pois)}")
    
    # Simple categories (grocery, healthcare, etc.)
    if not use_polygons:

        pois = pois[
            pois.geometry.type == "Point"
        ].copy()
        logger.info(
            f"Loaded {len(pois)} point POIs for category '{category_id}'"
        )
        

        return pois

    # Parks (or other polygon-based amenities)
    logger.info(f"Category '{category_id}' uses polygon processing")

    point_features = pois[
        pois.geometry.type == "Point"
    ].copy()

    polygon_features = pois[
        pois.geometry.type.isin(
            ["Polygon", "MultiPolygon"]
        )
    ].copy()
    logger.debug(
        f"Found {len(polygon_features)} polygon features for category '{category_id}'"
    )

    polygon_features["area_m2"] = polygon_features.area

    polygon_features = polygon_features[
        polygon_features["area_m2"] >= 5000
    ]
    logger.info(
        f"Filtered to {len(polygon_features)} large polygons for '{category_id}'"
    )

    boundary_points = []

    for geom in polygon_features.geometry:

        boundary = geom.boundary

        if boundary.geom_type == "LineString":

            boundary_points.extend(
                create_points_along_line(
                    boundary,
                    spacing=50
                )
            )

        elif boundary.geom_type == "MultiLineString":

            for line in boundary.geoms:

                boundary_points.extend(
                    create_points_along_line(
                        line,
                        spacing=50
                    )
                )

    boundary_points_gdf = gpd.GeoDataFrame(
        geometry=boundary_points,
        crs=target_crs
    )
    logger.debug(
        f"Created {len(boundary_points_gdf)} sampled boundary points for '{category_id}'"
    )

    combined = pd.concat(
        [
            point_features[["geometry"]],
            boundary_points_gdf
        ],
        ignore_index=True
    )
    logger.info(
        f"Returning {len(combined)} total POI geometries for category '{category_id}'"
    )

    return gpd.GeoDataFrame(
        combined,
        geometry="geometry",
        crs=target_crs
    )


In [7]:
def compute_accessibility(grid, graph, pois, category_id, target_crs):
    """
    Computes network distance from each hex cell to nearest POI
    """

    logger.info(f"Computing accessibility for '{category_id}'")
    centroids = grid.copy()
    centroids["geometry"] = centroids.geometry.centroid
    centroids = centroids.set_geometry("geometry")
    centroids = centroids.to_crs(target_crs)
    logger.debug(
        f"Centroids prepared: {len(centroids)} features; graph CRS={target_crs}"
    )
    
    nodes, edges = ox.graph_to_gdfs(graph)
    logger.debug(
        f"Graph converted to GeoDataFrames: {len(nodes)} nodes, {len(edges)} edges"
    )
    
    centroids["nearest_node"] = ox.distance.nearest_nodes(
        graph,
        X=centroids.geometry.x,
        Y=centroids.geometry.y
    )
    
    pois["nearest_node"] = ox.distance.nearest_nodes(
        graph,
        X=pois.geometry.x,
        Y=pois.geometry.y
    )
    logger.debug(
        f"Nearest nodes matched for {len(pois)} POIs"
    )
    
    grocery_nodes = list(pois["nearest_node"].unique())
    logger.info(
        f"Using {len(grocery_nodes)} unique source nodes for '{category_id}'"
    )
    
    distances = nx.multi_source_dijkstra_path_length(
        graph,
        sources=grocery_nodes,
        weight="length"
    )

    walk_col = f"{category_id}_walk_min"
    grid = grid.set_geometry("geometry")
    
    grid = grid.drop(columns=[
        col for col in grid.columns
        if col != "geometry" and str(grid[col].dtype) == "geometry"
    ])
    logger.debug(
        f"Cleaned up intermediate geometry columns for '{category_id}'"
    )

    grid[walk_col] = centroids["nearest_node"].map(distances) / 72
    logger.info(
        f"Computed walk duration field '{walk_col}' for '{category_id}'"
    )

    return grid


In [8]:
import gzip


def export_results(city_name, grid):
    """
    Writes the accessibility grid with only geometry and final accessibility fields.
    Outputs a GeoPackage, plain GeoJSON, and gzipped GeoJSON file.
    """

    logger.info(f"Exporting results for {city_name}")

    export_columns = [
        col for col in grid.columns
        if str(col) == "geometry" or str(col).endswith("_walk_min")
    ]
    export_grid = grid[export_columns].copy()

    gpkg_file = f"{city_name.lower().replace(' ', '_')}_accessibility.gpkg"
    export_grid.to_file(
        gpkg_file,
        layer="accessibility",
        driver="GPKG"
    )
    logger.info(f"GeoPackage written: {gpkg_file}")

    geojson_file = f"{city_name.lower().replace(' ', '_')}_accessibility.geojson"
    export_grid.to_crs("EPSG:4326").to_file(
        geojson_file,
        driver="GeoJSON"
    )
    logger.info(f"GeoJSON written: {geojson_file}")

    gzipped_file = f"{geojson_file}.gz"
    with open(geojson_file, "rb") as src, gzip.open(gzipped_file, "wb") as dst:
        dst.writelines(src)
    logger.info(f"Gzipped GeoJSON written: {gzipped_file}")

    return gpkg_file, geojson_file, gzipped_file


In [ ]:
for city in cities:
    logger.info(f"Processing city: {city}")
    
    boundary, G, target_crs = load_city(city)

    grid = create_hex_grid(boundary, cell_size, target_crs)

    for category in categories:
        logger.info(f"Starting category: {category['id']}")

        pois = load_pois(city, category, target_crs)

        grid = compute_accessibility(
            grid=grid,
            graph=G,
            pois=pois,
            category_id=category["id"],
            target_crs=target_crs
        )

    export_results(city, grid)


2026-06-15 12:36:07,247 INFO Processing city: Adelaide, South Australia, Australia
2026-06-15 12:36:07,248 INFO Loading city data for Adelaide, South Australia, Australia
2026-06-15 12:38:13,596 INFO City boundary cleaned by removing water areas
2026-06-15 12:38:13,600 INFO Creating hex grid for boundary with cell size 75 meters
2026-06-15 12:46:45,624 INFO Hex grid created and centroids computed
2026-06-15 12:46:45,625 INFO Starting category: grocery
2026-06-15 12:46:45,625 INFO Loading POIs for category 'grocery'
2026-06-15 12:46:45,807 INFO Loaded 247 point POIs for category 'grocery'
2026-06-15 12:46:45,808 INFO Computing accessibility for 'grocery'
2026-06-15 12:46:58,452 INFO Using 242 unique source nodes for 'grocery'
2026-06-15 12:46:59,819 INFO Computed walk duration field 'grocery_walk_min' for 'grocery'
2026-06-15 12:46:59,987 INFO Starting category: healthcare
2026-06-15 12:46:59,988 INFO Loading POIs for category 'healthcare'
2026-06-15 12:47:00,225 INFO Loaded 295 point P